# Zone Labelling (simple, single cell type)

Simplified version of `05_zone_labelling.ipynb`: no cell-type-grouping JSON, no looping
over multiple cell types, no separate "minimal" adata copy. You give it:
- **one query adata**, already subset to a single cell type
- **one reference adata**, the annotated zoning reference for that same cell type

and it maps zone labels from the reference onto the query, adding them directly as new
`.obs` columns on the query object — which is then saved back **in place**
(`QUERY_ADATA_PATH` is overwritten; there is no separate `_zoned.h5ad` file, since the
query already contains a single cell type).

**Requirements**
- Query must be `.h5ad`, with raw counts in `.X` (they get copied to `.layers["counts"]`)
- Reference must be `.h5ad`, with:
    - raw counts in `.X`
    - an `.obs` column with zone labels (`ZONE_LABELS_COL_NAME_IN_REF` below)
    - `var_names` = gene symbols

If the query's `var_names` are not already gene symbols (e.g. Ensembl IDs), they are
mapped to symbols via a BioMart lookup table before mapping.

**Outputs** (written to `WORKING_FOLDER`, reused across runs against the same reference):
- `precomputed_stats.h5`, `reference_markers.h5`, `query_markers.json`: reference-only, reusable across queries
- `mapping_output.csv` / `mapping_output.json`: mapping result for the last query run (overwritten each run, copied to `OUTPUT_FOLDER` for safekeeping)

In [1]:
%load_ext autoreload
%autoreload 2

# MUST BE FIRST - before any imports from cell_type_mapper --> error if FromSpecifiedMarkersRunner run with GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import json
import shutil
import subprocess

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from dotenv import load_dotenv; load_dotenv()

ZONE_LABELLING_SCRIPT = os.getenv("ZONE_LABELLING_SCRIPT")  # bash script that runs the mapping

## User config

Edit these paths and re-run the whole notebook.

In [2]:
# ATTENTION: query must already be subset to a single cell type.
# Zone labels get added to this file's .obs and it is SAVED BACK IN PLACE at the end
# (no separate "_zoned" copy is created).
CT ="Matrix_D1"
QUERY_ADATA_PATH = f"/home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/{CT}/{CT}.h5ad"

# ATTENTION: reference adata for the SAME cell type, with raw counts in .X and
# a zone-label column in .obs (see ZONE_LABELS_COL_NAME_IN_REF)
REFERENCE_ADATA_PATH = "/home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/STR_D1_Matrix_MSN.h5ad"

# obs col in the reference adata that contains the zone labels to map
ZONE_LABELS_COL_NAME_IN_REF = "Final_Zone_Assignments_No_Smooth"

# obs col name(s) to write the mapped zone / probability into on the query
OBS_COL_ZONE = "zone"

# lookup table to convert query var_names (e.g. Ensembl IDs) to gene symbols, if needed
BIOMART_GENE_MAP_PATH = "/home/gdallagl/myworkdir/XDP/data/BioMart/ensamble-name_biomart.txt"

# working folder: reused across queries mapped against the same reference
# (precomputed_stats / reference_markers / query_markers are only computed once per reference)
ref_name = os.path.splitext(os.path.basename(REFERENCE_ADATA_PATH))[0]
WORKING_FOLDER = os.path.join(os.path.dirname(REFERENCE_ADATA_PATH), "zoning_stats", ref_name)
OUTPUT_FOLDER = os.path.join(os.path.dirname(QUERY_ADATA_PATH), "zoning_output", ref_name)
os.makedirs(WORKING_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Query (in-place):", QUERY_ADATA_PATH)
print("Reference:       ", REFERENCE_ADATA_PATH)
print("Working folder:  ", WORKING_FOLDER)
print("Output folder:   ", OUTPUT_FOLDER)

Query (in-place): /home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/Matrix_D1/Matrix_D1.h5ad
Reference:        /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/STR_D1_Matrix_MSN.h5ad
Working folder:   /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/zoning_stats/STR_D1_Matrix_MSN
Output folder:    /home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/Matrix_D1/zoning_output/STR_D1_Matrix_MSN


## Load query and map gene IDs to symbols

In [3]:
query = sc.read_h5ad(QUERY_ADATA_PATH)
print(query)

assert "counts" in query.layers.keys(), "query must have a layer named 'counts' with raw counts"
assert "gene_symbol" in query.var.columns, "query must have a var column named 'gene_symbol' with gene symbols"


AnnData object with n_obs × n_vars = 441953 × 38095
    obs: 'tissue', 'ct_for_deg', 'prefix', 'cell_barcode', 'expression_doublet', 'num_genic_reads', 'num_transcripts', 'num_genes', 'num_retained_transcripts', 'pct_coding', 'pct_utr', 'pct_intergenic', 'pct_intronic', 'pct_mt', 'frac_contamination', 'donor_id', 'biobank', 'cohort', 'age', 'race', 'pmi_hr', 'sex', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'mmc_neighborhood_name', 'mmc_neighborhood_bootstrapping_probability', 'mmc_class_name', 'mmc_class_bootstrapping_probability', 'mmc_subclass_name', 'mmc_subclass_bootstrapping_probability', 'mmc_group_name', 'mmc_group_bootstrapping_probability', 'mmc_cluster_name', 'mmc_cluster_bootstrapping_probability', 'village', 'brain_region_abbreviation', 'donor_outlier', 'sample_outlier', 'gex_donor_outlier', 'gex_donor_celltype_outlier'
    var: 'gene_symbol', 'ensembl_id'
    layers: 'counts'


## Check reference

In [4]:
ref = sc.read_h5ad(REFERENCE_ADATA_PATH, backed="r")

has_zone_col = ZONE_LABELS_COL_NAME_IN_REF in ref.obs.columns
print(f"Reference has zone column {ZONE_LABELS_COL_NAME_IN_REF}: {has_zone_col}")
assert has_zone_col, f"Reference is missing zone column {ZONE_LABELS_COL_NAME_IN_REF}"

query_genes = set(query.var["gene_symbol"])
ref_genes = set(ref.var_names)
overlap = len(query_genes & ref_genes) / len(ref_genes)
print(f"Gene overlap (query vs reference): {overlap:.1%}")

del ref
assert overlap >= 0.8, f"Gene overlap too low ({overlap:.1%}). Check that query var gene_symbol matches reference var_names."

Reference has zone column Final_Zone_Assignments_No_Smooth: True
Gene overlap (query vs reference): 97.5%


## Write query for mapping and run the mapping script

Reuses the `query` object already in memory instead of building a separate minimal
AnnData: gene symbols and raw counts are already there, so we just swap `.var_names`
and `.X` to what the mapping script expects, write it out, then swap back.

In [5]:
subprocess.run(["chmod", "+x", ZONE_LABELLING_SCRIPT], check=True)

cmd = [
    ZONE_LABELLING_SCRIPT,
    "-r", REFERENCE_ADATA_PATH,
    "-q", QUERY_ADATA_PATH,
    "-o", WORKING_FOLDER,
    "-z", ZONE_LABELS_COL_NAME_IN_REF,
    # "-c",  # pass -c to force recomputing precomputed_stats/reference_markers/query_markers
]

print("Running mapping script...")
print(" ".join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Mapping script failed")

print("Mapping complete")

shutil.copy2(os.path.join(WORKING_FOLDER, "mapping_output.csv"), os.path.join(OUTPUT_FOLDER, "mapping_output.csv"))
shutil.copy2(os.path.join(WORKING_FOLDER, "mapping_output.json"), os.path.join(OUTPUT_FOLDER, "mapping_output.json"))
print(f"Results copied to: {OUTPUT_FOLDER}")

Running mapping script...
/home/gdallagl/myworkdir/XDP/utils/STR_cell_types_annotation/run_mapmycells_zones.sh -r /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/STR_D1_Matrix_MSN.h5ad -q /home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/Matrix_D1/Matrix_D1.h5ad -o /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/zoning_stats/STR_D1_Matrix_MSN -z Final_Zone_Assignments_No_Smooth
/home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad
make output directory
Reference Directory: /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad
Query Directory: /home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/Matrix_D1/Matrix_D1.h5ad
Output Directory: /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/zoning_stats/STR_D1_Matrix_MSN
get precomputed stats
pivoting /home/gdallagl/myworkdir/XDP/data/STR_zonation_references_h5ad/STR_D1_Matrix_MSN.h5ad -> STR_D1_Matrix_MSN_csr_w94vq5yu.h5ad
joining took 7.304863e+00 seconds
finally proc

## Add zone labels to query (in place)

In [6]:
mapping_df = pd.read_csv(os.path.join(OUTPUT_FOLDER, "mapping_output.csv"), comment="#")
mapping_df.set_index("cell_id", inplace=True)
mapping_df.index = mapping_df.index.astype(str)
print(f"Loaded mapping for {len(mapping_df)} cells")

source_zone_col = f"{ZONE_LABELS_COL_NAME_IN_REF}_name"  # defined by the mapping script
source_prob_col = f"{ZONE_LABELS_COL_NAME_IN_REF}_bootstrapping_probability"

target_zone_col = OBS_COL_ZONE
target_prob_col = f"{OBS_COL_ZONE}_probability"

query.obs[target_zone_col] = np.nan
query.obs[target_prob_col] = np.nan

common_cells = query.obs_names.intersection(mapping_df.index)
query.obs.loc[common_cells, target_zone_col] = mapping_df.loc[common_cells, source_zone_col].values
query.obs.loc[common_cells, target_prob_col] = mapping_df.loc[common_cells, source_prob_col].values

# Cast to an ordered categorical (1, 2, 3, ...)
valid_zones = query.obs[target_zone_col].dropna()
n_zones = int(valid_zones.max())
query.obs[target_zone_col] = pd.Categorical(
    query.obs[target_zone_col].astype("Int64").astype(str),
    categories=[str(i) for i in range(1, n_zones + 1)],
    ordered=True,
)

print(query.obs[target_zone_col].value_counts(dropna=False).sort_index())

Loaded mapping for 441953 cells
zone
1     81771
2     84904
3     40359
4    139540
5     54184
6     41195
Name: count, dtype: int64


## Save (in place)

Overwrites `QUERY_ADATA_PATH` with the zone columns added — no separate `_zoned.h5ad` file.

In [7]:
query.write(QUERY_ADATA_PATH)
print(f"Saved: {QUERY_ADATA_PATH}")

Saved: /home/gdallagl/myworkdir/XDP/data/BICAN/whole_BICAN_ezra/Matrix_D1/Matrix_D1.h5ad


## Quick sanity plot

In [8]:
basis = "spatial" if "spatial" in query.obsm else ("X_umap" if "X_umap" in query.obsm else None)
if basis is not None:
    sc.pl.embedding(query, basis=basis, color=[OBS_COL_ZONE, f"{OBS_COL_ZONE}_probability"], size=20)
else:
    print("No 'spatial' or 'X_umap' in .obsm, skipping plot.")

No 'spatial' or 'X_umap' in .obsm, skipping plot.
